In [2]:
from tqdm import tqdm
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

combined_novels_nyt_df = pd.read_csv('combined_novels_with_entities_and_cleaned_text.csv')
# Drop rows with missing cleaned text
tfidf_df = combined_novels_nyt_df.dropna(subset=['cleaned_text']).copy()

# min_df=2: ignore terms appearing in fewer than 2 documents (typos, OCR errors)
# max_df=0.75: ignore terms appearing in more than 75% of documents (too common to be distinctive)
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000,
    min_df=2,
    max_df=0.75,
    
)

tfidf_matrix = vectorizer.fit_transform(tfidf_df['cleaned_text'])
feature_names = vectorizer.get_feature_names_out()

print(f"Matrix shape: {tfidf_matrix.shape}")
print(f"  {tfidf_matrix.shape[0]} documents × {tfidf_matrix.shape[1]} terms")

Matrix shape: (175, 5000)
  175 documents × 5000 terms


In [6]:
import altair as alt

# Use all genres — only exclude rows with no genre label at all
genre_df = tfidf_df.dropna(subset=['genre']).copy()
all_genres = genre_df['genre'].unique().tolist()

# Re-index the matrix rows to match genre_df
genre_indices = [tfidf_df.index.get_loc(i) for i in genre_df.index]
tfidf_matrix_genres = tfidf_matrix[genre_indices]

# Compute genre centroids
genre_centroids = {}
for genre in all_genres:
    mask = (genre_df['genre'] == genre).values
    genre_centroids[genre] = np.asarray(tfidf_matrix_genres[mask].mean(axis=0))

def top_centroid_terms(genre, n=12):
    centroid = genre_centroids[genre].flatten()
    top_idx = centroid.argsort()[::-1][:n]
    return [(feature_names[i], float(centroid[i])) for i in top_idx]

fingerprint_rows = []
for genre in all_genres:
    for term, score in top_centroid_terms(genre):
        fingerprint_rows.append({'genre': genre, 'term': term, 'centroid_score': score})

fingerprint_df = pd.DataFrame(fingerprint_rows)

In [7]:
# fingerprint_df = pd.read_csv('../../assets/files/combined_novels_genre_fingerprint.csv')

alt.Chart(fingerprint_df).mark_bar().encode(
    x=alt.X('centroid_score:Q', title='Mean TF-IDF Score'),
    y=alt.Y('term:N', sort='-x'),
    color='genre:N',
    facet=alt.Facet('genre:N', columns=2),
    tooltip=['genre', 'term', 'centroid_score']
).properties(width=300, height=220).resolve_scale(y='independent')

alt.Chart(...)

In [5]:
import re
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def parse_entity_list(val):
    """Handle both real lists (in-memory) and stringified lists (loaded from CSV)."""
    if isinstance(val, list):
        return val
    if isinstance(val, str) and val.startswith('['):
        try:
            return ast.literal_eval(val)
        except Exception:
            return []
    return []

def remove_person_tokens(row):
    """Strip person entity tokens from a novel's cleaned text."""
    text = str(row['cleaned_text'])
    entities = parse_entity_list(row['person_entities'])

    if not entities:
        return text

    # Split multi-word names ("Elizabeth Bennet" → {"elizabeth", "bennet"})
    # Skip tokens of 2 chars or fewer to avoid clipping common words
    name_tokens = {
        token.lower()
        for entity in entities
        for token in re.findall(r'[a-zA-Z]+', str(entity))
        if len(token) > 2
    }

    filtered = [
        w for w in re.findall(r'[a-zA-Z]+', text)
        if w.lower() not in name_tokens
    ]
    return ' '.join(filtered)

# Drop rows with missing cleaned text or person_entities
tfidf_df = combined_novels_nyt_df.dropna(subset=['cleaned_text', 'person_entities']).copy()

tfidf_df['cleaned_text_no_persons'] = tfidf_df.apply(remove_person_tokens, axis=1)

vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.75,
    stop_words='english'
)

tfidf_matrix = vectorizer.fit_transform(tfidf_df['cleaned_text_no_persons'])
feature_names = vectorizer.get_feature_names_out()

print(f"Matrix shape: {tfidf_matrix.shape}")
print(f"  {tfidf_matrix.shape[0]} documents × {tfidf_matrix.shape[1]} terms")

Matrix shape: (175, 5000)
  175 documents × 5000 terms


In [8]:
from sklearn.metrics.pairwise import cosine_similarity

# Use all genres — only exclude rows with no genre label at all
genre_df = tfidf_df[tfidf_df.genre != 'na'].dropna(subset=['genre']).copy()
all_genres = genre_df['genre'].unique().tolist()

# Re-index the matrix rows to match genre_df
genre_indices = [tfidf_df.index.get_loc(i) for i in genre_df.index]
tfidf_matrix_genres = tfidf_matrix[genre_indices]

# Compute genre centroids
genre_centroids = {}
for genre in all_genres:
    mask = (genre_df['genre'] == genre).values
    genre_centroids[genre] = np.asarray(tfidf_matrix_genres[mask].mean(axis=0))

def top_centroid_terms(genre, n=12):
    centroid = genre_centroids[genre].flatten()
    top_idx = centroid.argsort()[::-1][:n]
    return [(feature_names[i], float(centroid[i])) for i in top_idx]

# Isolate the unlabeled novels
unlabeled_df = tfidf_df[tfidf_df['genre'] == 'na'].copy()
print(f"{len(unlabeled_df)} novels with no genre label")

# Get their rows from the TF-IDF matrix
unlabeled_indices = [tfidf_df.index.get_loc(i) for i in unlabeled_df.index]
tfidf_matrix_unlabeled = tfidf_matrix[unlabeled_indices]

# Cosine similarity to each genre centroid
centroid_matrix = np.vstack([genre_centroids[g] for g in all_genres])
sim_scores = cosine_similarity(tfidf_matrix_unlabeled, centroid_matrix)
sim_df = pd.DataFrame(sim_scores, columns=all_genres, index=unlabeled_df.index)

# Add nearest genre and confidence as new columns
unlabeled_df['tfidf_genre_suggestion'] = sim_df.idxmax(axis=1)
unlabeled_df['tfidf_genre_confidence'] = sim_df.max(axis=1).round(4)

# Save top 5 distinctive terms per novel — context for the LLM step later
def get_top_terms_unlabeled(row_index, n=5):
    row = tfidf_matrix_unlabeled[row_index]
    top_idx = np.argsort(row.toarray()[0])[::-1][:n]
    return ', '.join(feature_names[i] for i in top_idx)

unlabeled_df['tfidf_top_terms'] = [
    get_top_terms_unlabeled(i) for i in range(tfidf_matrix_unlabeled.shape[0])
]

unlabeled_df[['title', 'genre', 'author', 'tfidf_genre_suggestion', 'tfidf_genre_confidence', 'tfidf_top_terms']].head(10)

94 novels with no genre label


,title,genre,author,tfidf_genre_suggestion,tfidf_genre_confidence,tfidf_top_terms
5,Wuthering Heights,na,Emily Brontë,romance,0.6006,"mrs, papa, cousin, upstairs, ye"
7,Moby Dick,na,Herman Melville,action,0.3857,"sperm, ye, chapter, captain, aye"
8,The Scarlet Letter,na,Nathaniel Hawthorne,history,0.2966,"pearl, thou, minister, scarlet, thee"
11,A Christmas Carol,na,Charles Dickens,romance,0.3001,"christmas, mrs, joe, fred, ha"
14,Little Women,na,Louisa May Alcott,romance,0.0953,"jo, mrs, illustration, aunt, fred"
19,Crime and Punishment,na,Fyodor Dostoyevsky,political,0.3047,"roubles, petersburg, landlady, police, sofa"
20,Madame Bovary: Patterns of Provincial life,na,Gustave Flaubert,bildung,0.4363,"madame, chemist, hippolyte, francs, notary"
21,Dracula,na,Bram Stoker,political,0.2964,"arthur, dr, professor, morris, whilst"
25,Les Misérables,na,Victor Hugo,history,0.2143,"marius, rue, chapter, th, barricade"
26,The Secret Garden,na,Frances Hodgson Burnett,romance,0.1346,"colin, th, robin, mrs, moor"


In [9]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

count_vectorizer = CountVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.75,
    stop_words='english'
)

count_matrix = count_vectorizer.fit_transform(genre_df['cleaned_text_no_persons'])
count_feature_names = count_vectorizer.get_feature_names_out()

print(f"Count matrix shape: {count_matrix.shape}")
print(f"  {count_matrix.shape[0]} documents × {count_matrix.shape[1]} terms")

Count matrix shape: (81, 5000)
  81 documents × 5000 terms


In [12]:
N_TOPICS = len(all_genres)  # one topic per genre as a starting point

lda_model = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=42,
    max_iter=20,
    learning_method='batch'
)

# This will take a minute or two
lda_matrix = lda_model.fit_transform(count_matrix)

print(f"LDA output shape: {lda_matrix.shape}")
print(f"  {lda_matrix.shape[0]} documents × {lda_matrix.shape[1]} topics")

LDA output shape: (81, 11)
  81 documents × 11 topics


In [13]:
def print_top_words(model, feature_names, n_top_words=12):
    for topic_idx, topic in enumerate(model.components_):
        top_idx = topic.argsort()[::-1][:n_top_words]
        top_words = [feature_names[i] for i in top_idx]
        print(f"Topic {topic_idx:2d}: {', '.join(top_words)}")

print_top_words(lda_model, count_feature_names)

Topic  0: mrs, miss, em, ma, caroline, didn, wouldn, london, rejoined, agnes, ha, couldn
Topic  1: thee, caesar, alexey, petronius, alexandrovitch, rome, christ, princess, rochester, chapter, dolly, christians
Topic  2: thou, thee, knight, se, thy, worship, jpg, curate, squire, errant, knights, duchess
Topic  3: captain, fang, bull, camp, dollars, gun, cave, ain, san, friday, london, em
Topic  4: island, engineer, captain, nautilus, granite, sailor, th, vessel, scout, rocks, coast, reporter
Topic  5: mrs, didn, isn, doesn, couldn, wouldn, tea, dr, inspector, paris, wasn, murmured
Topic  6: thy, thee, hath, hast, priest, christian, honour, prince, ay, lama, holy, thine
Topic  7: aramis, athos, madame, porthos, la, cardinal, paris, monseigneur, king, tr, france, duke
Topic  8: mrs, sha, nat, moscow, emperor, russian, nicholas, napoleon, colonel, den, petersburg, soldiers
Topic  9: mrs, uncle, dinah, arthur, wi, cousin, chapter, miss, th, farm, lad, honour
Topic 10: monte, cristo, odette,

In [14]:
topic_cols = [f'topic_{i}' for i in range(N_TOPICS)]
topics_df = pd.DataFrame(lda_matrix, columns=topic_cols, index=genre_df.index)

# Join topic weights back to novel metadata
novels_topics_df = genre_df[['title', 'author', 'genre', 'pub_year']].join(topics_df)

# Add the dominant topic as a single summary column
novels_topics_df['dominant_topic'] = topics_df.idxmax(axis=1)

print(f"Added {N_TOPICS} topic columns + 'dominant_topic' to {len(novels_topics_df)} novels")
novels_topics_df[['title', 'genre', 'dominant_topic'] + topic_cols[:3]].head()

Added 11 topic columns + 'dominant_topic' to 81 novels


,title,genre,dominant_topic,topic_0,topic_1,topic_2
0,Don Quixote,action,topic_2,0.000002,0.000002,0.937072
1,Alice's Adventures in Wonderland,fantasy,topic_5,0.104620,0.000064,0.190410
2,The Adventures of Tom Sawyer,action,topic_3,0.196207,0.000017,0.000017
3,Treasure Island,action,topic_3,0.041044,0.000020,0.000020
4,Pride and Prejudice,romance,topic_8,0.000869,0.000011,0.000011


In [16]:
import altair as alt

topic_genre_df = novels_topics_df.melt(
    id_vars=['genre'],
    value_vars=topic_cols,
    var_name='topic',
    value_name='weight'
)

topic_genre_means = (
    topic_genre_df
    .groupby(['genre', 'topic'])['weight']
    .mean()
    .reset_index()
)

# Normalize within each topic so genres are comparable
topic_genre_means['normalized'] = topic_genre_means.groupby('topic')['weight'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
)

# topic_genre_means = pd.read_csv('../../assets/files/combined_novels_topic_genre_means.csv')
alt.Chart(topic_genre_means).mark_rect().encode(
    x=alt.X('topic:N', title='LDA Topic'),
    y=alt.Y('genre:N', title='Genre'),
    color=alt.Color('normalized:Q',
                    scale=alt.Scale(scheme='oranges'),
                    title='Normalized Mean Weight'),
    tooltip=[
        alt.Tooltip('genre:N'),
        alt.Tooltip('topic:N'),
        alt.Tooltip('weight:Q', format='.3f', title='Mean Weight'),
        alt.Tooltip('normalized:Q', format='.2f', title='Normalized')
    ]
).properties(
    title='Topic-Genre Alignment (normalized within topic)',
    width=500,
    height=350
)

alt.Chart(...)

In [17]:
# Transform unlabeled novels through the already-fitted models
unlabeled_count_matrix = count_vectorizer.transform(unlabeled_df['cleaned_text_no_persons'])
unlabeled_lda_matrix = lda_model.transform(unlabeled_count_matrix)

# Add topic weight columns and dominant topic to unlabeled_df
unlabeled_topics_df = pd.DataFrame(
    unlabeled_lda_matrix, columns=topic_cols, index=unlabeled_df.index
)
unlabeled_df = unlabeled_df.join(unlabeled_topics_df)
unlabeled_df['dominant_topic'] = unlabeled_topics_df.idxmax(axis=1)

# Build a lookup of top words per topic — used as context in the LLM prompt
topic_top_words = {}
for topic_idx, topic in enumerate(lda_model.components_):
    top_idx = topic.argsort()[::-1][:8]
    topic_top_words[f'topic_{topic_idx}'] = ', '.join(count_feature_names[i] for i in top_idx)

print(f"unlabeled_df now has {unlabeled_df.shape[1]} columns")
unlabeled_df[['title', 'author', 'tfidf_genre_suggestion', 'dominant_topic']].head(10)

unlabeled_df now has 70 columns


,title,author,tfidf_genre_suggestion,dominant_topic
5,Wuthering Heights,Emily Brontë,romance,topic_0
7,Moby Dick,Herman Melville,action,topic_3
8,The Scarlet Letter,Nathaniel Hawthorne,history,topic_6
11,A Christmas Carol,Charles Dickens,romance,topic_0
14,Little Women,Louisa May Alcott,romance,topic_5
19,Crime and Punishment,Fyodor Dostoyevsky,political,topic_5
20,Madame Bovary: Patterns of Provincial life,Gustave Flaubert,bildung,topic_5
21,Dracula,Bram Stoker,political,topic_5
25,Les Misérables,Victor Hugo,history,topic_7
26,The Secret Garden,Frances Hodgson Burnett,romance,topic_5


In [18]:
import ollama

response = ollama.chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': 'Say hello in one sentence.'}]
)
print(response['message']['content'])

Hello!


In [19]:
all_genres

['action',
 'fantasy',
 'romance',
 'allegories',
 'bildung',
 'history',
 'horror',
 'political',
 'scifi',
 'mystery',
 'autobio']

In [20]:
import ollama

# Build the genre list from our actual dataset
genre_list = '\n'.join(f'- {g}' for g in sorted(all_genres))

def classify_cold(row):
    """Zero-shot genre classification — text only, no prior signals."""
    text = str(row['cleaned_text_no_persons'])
    if pd.isna(text) or len(text) < 100:
        return None
    passage = text[2000:3000]

    prompt = f"""
    You are a literary scholar classifying novels by genre.

	Novel: "{row['title']}" by {row['author']}

	Passage:{passage}

	Classify this novel into EXACTLY ONE of the following genres:{genre_list}

	Respond with ONLY the genre name from the list above. No explanation, no punctuation."""

    try:
        response = ollama.chat(
            model='llama3.2',
            messages=[{'role': 'user', 'content': prompt}]
        )
        return response['message']['content'].strip()
    except Exception as e:
        print(f"Error for {row['title']}: {e}")
        return None

# Test on a small sample first
sample = unlabeled_df.head(5).copy()
sample['llm_genre_cold'] = sample.apply(classify_cold, axis=1)
sample[['title', 'author', 'tfidf_genre_suggestion', 'llm_genre_cold']]

,title,author,tfidf_genre_suggestion,llm_genre_cold
5,Wuthering Heights,Emily Brontë,romance,romance
7,Moby Dick,Herman Melville,action,allegory
8,The Scarlet Letter,Nathaniel Hawthorne,history,allegory
11,A Christmas Carol,Charles Dickens,romance,allegory
14,Little Women,Louisa May Alcott,romance,bildung


In [21]:
def classify_cold(row):
    """Zero-shot genre classification — text only, no prior signals."""
    text = str(row['cleaned_text_no_persons'])
    if pd.isna(text) or len(text) < 100:
        return None, None
    passage = text[2000:3000]

    prompt = f"""You are a literary scholar classifying novels by genre.

Novel: "{row['title']}" by {row['author']}

Passage:
{passage}

Classify this novel into EXACTLY ONE of the following genres:
{genre_list}

Respond in this EXACT format:
GENRE: [genre name from the list above]
RATIONALE: [one sentence explaining what in the passage led to this classification]"""

    try:
        response = ollama.chat(
            model='llama3.2',
            messages=[{'role': 'user', 'content': prompt}]
        )
        raw = response['message']['content'].strip()
        genre, rationale = None, None
        for line in raw.split('\n'):
            if line.startswith('GENRE:'):
                genre = line.partition(':')[2].strip()
            elif line.startswith('RATIONALE:'):
                rationale = line.partition(':')[2].strip()
        return genre, rationale
    except Exception as e:
        print(f"Error for {row['title']}: {e}")
        return None, None

sample = unlabeled_df.head(5).copy()
results = sample.apply(classify_cold, axis=1, result_type='expand')
sample[['llm_genre_cold', 'llm_genre_cold_rationale']] = results

sample[['title', 'author', 'tfidf_genre_suggestion', 'llm_genre_cold', 'llm_genre_cold_rationale']]

,title,author,tfidf_genre_suggestion,llm_genre_cold,llm_genre_cold_rationale
5,Wuthering Heights,Emily Brontë,romance,bildung,The passage's focus on describing the exterior...
7,Moby Dick,Herman Melville,action,allegory,The novel's use of abstract and symbolic langu...
8,The Scarlet Letter,Nathaniel Hawthorne,history,allegory,The detailed and poetic illustrations accompan...
11,A Christmas Carol,Charles Dickens,romance,allegory,The passage's focus on the protagonist's inner...
14,Little Women,Louisa May Alcott,romance,bildung,The passage appears to describe a scene of fam...


In [22]:
sample[['title', 'author', 'tfidf_genre_suggestion', 'llm_genre_cold', 'llm_genre_cold_rationale']].to_dict(orient='records')

[{'title': 'Wuthering Heights',
  'author': 'Emily Brontë',
  'tfidf_genre_suggestion': 'romance',
  'llm_genre_cold': 'bildung',
  'llm_genre_cold_rationale': "The passage's focus on describing the exterior of a remote and atmospheric location, and the attention to detail about its architecture and natural surroundings, suggests a emphasis on education or coming-of-age themes, which is characteristic of the bildungsroman genre."},
 {'title': 'Moby Dick',
  'author': 'Herman Melville',
  'tfidf_genre_suggestion': 'action',
  'llm_genre_cold': 'allegory',
  'llm_genre_cold_rationale': "The novel's use of abstract and symbolic language, such as the white whale Moby Dick serving as a metaphor for the unknowable and elusive nature of life, is characteristic of allegorical works that convey moral or philosophical messages through narrative."},
 {'title': 'The Scarlet Letter',
  'author': 'Nathaniel Hawthorne',
  'tfidf_genre_suggestion': 'history',
  'llm_genre_cold': 'allegory',
  'llm_gen

In [23]:
def classify_with_signals(row):
    """Genre classification informed by TF-IDF vocabulary and LDA topic signals."""
    text = str(row['cleaned_text_no_persons'])
    if pd.isna(text) or len(text) < 100:
        return None, None
    passage = text[2000:3000]

    dominant = row['dominant_topic']
    topic_words = topic_top_words.get(dominant, 'unknown')

    prompt = f"""You are a literary scholar classifying novels by genre.

Novel: "{row['title']}" by {row['author']}

Two computational methods have already analyzed this novel:
- Vocabulary analysis (TF-IDF) suggests: {row['tfidf_genre_suggestion']} (confidence: {row['tfidf_genre_confidence']:.2f})
  Most distinctive terms: {row['tfidf_top_terms']}
- Topic modeling (LDA) found this thematic cluster: {topic_words}

Now read this passage and make your own judgment:
{passage}

Based on the passage AND the signals above, classify this novel into EXACTLY ONE of these genres:
{genre_list}

If the passage confirms the vocabulary analysis, use that genre. If the passage makes a stronger case for something different, use the passage.

Respond in this EXACT format:
GENRE: [genre name from the list above]
RATIONALE: [one sentence explaining what in the passage led to this classification]"""

    try:
        response = ollama.chat(
            model='llama3.2',
            messages=[{'role': 'user', 'content': prompt}]
        )
        raw = response['message']['content'].strip()
        genre, rationale = None, None
        for line in raw.split('\n'):
            if line.startswith('GENRE:'):
                genre = line.partition(':')[2].strip()
            elif line.startswith('RATIONALE:'):
                rationale = line.partition(':')[2].strip()
        return genre, rationale
    except Exception as e:
        print(f"Error for {row['title']}: {e}")
        return None, None

results = sample.apply(classify_with_signals, axis=1, result_type='expand')
sample[['llm_genre_signals', 'llm_genre_signals_rationale']] = results

sample[['title', 'author', 'tfidf_genre_suggestion', 'llm_genre_cold', 'llm_genre_signals', 'llm_genre_signals_rationale']]

,title,author,tfidf_genre_suggestion,llm_genre_cold,llm_genre_signals,llm_genre_signals_rationale
5,Wuthering Heights,Emily Brontë,romance,bildung,romance,The passage's focus on descriptive language ab...
7,Moby Dick,Herman Melville,action,allegory,allegories,The passage's repetitive structure and lack of...
8,The Scarlet Letter,Nathaniel Hawthorne,history,allegory,allegory,"The passage's themes of redemption, forgivenes..."
11,A Christmas Carol,Charles Dickens,romance,allegory,allegory,The passage's focus on the main character's in...
14,Little Women,Louisa May Alcott,romance,bildung,romance,The passage's focus on Jo and her interactions...


In [24]:
sample['cold_agrees_tfidf'] = sample['llm_genre_cold'] == sample['tfidf_genre_suggestion']
sample['signals_agrees_tfidf'] = sample['llm_genre_signals'] == sample['tfidf_genre_suggestion']
sample['methods_agree'] = sample['llm_genre_cold'] == sample['llm_genre_signals']

print(f"Cold LLM agrees with TF-IDF suggestion:            {sample['cold_agrees_tfidf'].mean():.0%}")
print(f"Signal-informed LLM agrees with TF-IDF suggestion: {sample['signals_agrees_tfidf'].mean():.0%}")
print(f"Both LLM approaches agree with each other:         {sample['methods_agree'].mean():.0%}")

# Show disagreements with rationale — these are the interesting cases
disagreements = sample[~sample['methods_agree']][[
    'title', 'tfidf_genre_suggestion',
    'llm_genre_cold',
    'llm_genre_signals', 'llm_genre_signals_rationale'
]]

if len(disagreements) > 0:
    print(f"\n{len(disagreements)} novel(s) where cold and signal-informed disagree:\n")
    for _, row in disagreements.iterrows():
        print(f"  {row['title']}")
        print(f"    TF-IDF suggestion : {row['tfidf_genre_suggestion']}")
        print(f"    Cold LLM          : {row['llm_genre_cold']}")
        print(f"    Signal-informed   : {row['llm_genre_signals']}")
        print(f"    Rationale         : {row['llm_genre_signals_rationale']}")
        print()
else:
    print("\nAll approaches agree on this sample.")

Cold LLM agrees with TF-IDF suggestion:            0%
Signal-informed LLM agrees with TF-IDF suggestion: 40%
Both LLM approaches agree with each other:         40%

3 novel(s) where cold and signal-informed disagree:

  Wuthering Heights
    TF-IDF suggestion : romance
    Cold LLM          : bildung
    Signal-informed   : romance
    Rationale         : The passage's focus on descriptive language about a rural setting and the protagonist's emotional response upon arriving at Mr. S's dwelling, particularly the mention of "pious ejaculation" and admiring the "grotesque carving", suggests a romantic and sentimental tone consistent with Emily Brontë's novel "Wuthering Heights".

  Moby Dick
    TF-IDF suggestion : action
    Cold LLM          : allegory
    Signal-informed   : allegories
    Rationale         : The passage's repetitive structure and lack of descriptive narrative detail suggest a focus on themes and symbolism over action or plot, aligning with the characteristic features 

In [25]:
from tqdm import tqdm
tqdm.pandas(desc="Classifying novels")

unlabeled_df['llm_genre'] = unlabeled_df.progress_apply(classify_with_signals, axis=1)

print(f"Classified: {unlabeled_df['llm_genre'].notna().sum()} / {len(unlabeled_df)} novels")
unlabeled_df[['title', 'author', 'tfidf_genre_suggestion', 'dominant_topic', 'llm_genre']].head(20)

Classifying novels: 100%|██████████| 94/94 [01:04<00:00,  1.45it/s]

Classified: 94 / 94 novels


,title,author,tfidf_genre_suggestion,dominant_topic,llm_genre
5,Wuthering Heights,Emily Brontë,romance,topic_0,"(romance, The passage's focus on atmosphere an..."
7,Moby Dick,Herman Melville,action,topic_3,"(action, The abundance of verbs such as ""watch..."
8,The Scarlet Letter,Nathaniel Hawthorne,history,topic_6,"(allegories, The presence of vivid and symboli..."
11,A Christmas Carol,Charles Dickens,romance,topic_0,"(allegory, The passage's focus on the main cha..."
14,Little Women,Louisa May Alcott,romance,topic_5,"(romance, The passage's focus on domestic life..."
19,Crime and Punishment,Fyodor Dostoyevsky,political,topic_5,"(psychological, The passage highlights the pro..."
20,Madame Bovary: Patterns of Provincial life,Gustave Flaubert,bildung,topic_5,"(bildung, The detailed and observational descr..."
21,Dracula,Bram Stoker,political,topic_5,"(horror, The passage's focus on dark and foreb..."
25,Les Misérables,Victor Hugo,history,topic_7,"(history, The explicit use of historical dates..."
26,The Secret Garden,Frances Hodgson Burnett,romance,topic_5,"(Bildung, The passage highlights the protagoni..."
